In [44]:
"""
LangGraph implementation for DeepAgent style Chinese writing agent.

Features:
1. YAML-driven routing after intent node.
2. YAML-driven flow edges, so different intents can trigger different execution paths.
3. LangGraph checkpointer memory with thread_id.
4. Optimized DeepAgentState: separates persistent memory from intermediate scratch state.
5. OpenAI-compatible LLM client for vLLM / SGLang / LMDeploy / OpenAI gateways.

Run:
    pip install openai langgraph langchain-core json-repair pyyaml

    export OPENAI_API_BASE="http://127.0.0.1:8000/v1"
    export OPENAI_API_KEY="EMPTY"
    export OPENAI_MODEL="qwen"

    python deepagent_langgraph_yaml_memory.py "帮我写一份项目进展文档" --thread-id user_001

Optional:
    export DEEPAGENT_NODE_MODELS='{"intent":"qwen3-8b","planner":"qwen3-32b","draft":"qwen3-32b"}'
    export DEEPAGENT_NODE_TEMPERATURES='{"intent":0.0,"planner":0.2,"draft":0.6}'
"""
import yaml
from json_repair import repair_json

from utils import *
from state import *
from openai import OpenAI
from llm_client import LLMClient
from typing import Any, Callable, Dict, List, Mapping, Optional, Sequence, Literal

# 相关配置
from flow_config import FlowConfig
from prompt_registry import PromptRegistry

# 相关执行class
from nn_recall_passk import recall_passk_function

try:
    from langgraph.checkpoint.memory import InMemorySaver
    from langgraph.graph import END, START, StateGraph
    from langgraph.graph.message import add_messages
except ImportError as exc:  # pragma: no cover
    raise RuntimeError(
        "Missing langgraph dependencies. Please run: "
        "pip install langgraph langchain-core"
    ) from exc

from llm_summary_optimizer import LLMDescriptionSummary
from llm_description_judge import LLMDescriptionJudge
from enum import Enum

class Const(Enum):
    PATH = '../data/data_not_import/'
    HOST = "tianchi-proxy.baidu-int.com"
    APPID = 'app-RNgOjXzL'
    DEFAULT_MODEL = "deepseek-v4-flash"


/root/paddlejob/workspace/env_run/output/zacharychu/miniconda3/envs/server/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:11<00:00, 35.88it/s]


In [ ]:

class ToolOptimizerGraph:
    def __init__(
        self,
        resource_id: str,
        llm_client: Optional[LLMClient] = None,
        prompt_path: str = "../config/prompts.json",
        flow_config_path: str = "../config/agent_config.yaml",
        tools_description_path: str = "../data/summary/tool_descriptions.json",
        test_data_path: str = "../data/summary/query.json",
        checkpointer: Optional[Any] = None,
    ) -> None:
        # 设置通用的 参数
        self.prompts = PromptRegistry(prompt_path).get_promts()
        self.flow_config = FlowConfig(flow_config_path)
        self.checkpointer = checkpointer or InMemorySaver()
        self.last_tools_description_path = tools_description_path
        # 设置 tool resource_id
        self.resource_id = resource_id

        # 加载tools 这个列表
        self.tools_dict = load_tools_from_json(self.last_tools_description_path)

        # 加载测试集
        # test_data_path: str = "../data/summary/query.json"
        # load_tools_from_json(test_data_path)#
        self.query_good_all_dict = load_tools_from_json(test_data_path)
        self.query_good_dict = self.query_good_all_dict[resource_id]

        # node -> model
        self.node_model_map: Dict[str, str] = {} 
        if self.flow_config.node_model_map:
            self.node_model_map.update({str(k): str(v) for k, v in self.flow_config.node_model_map.items()})

        # node -> temperature
        self.node_temperature_map: Dict[str, float] =  {} 
        if self.flow_config.node_temperature_map:
            self.node_temperature_map.update({str(k): float(v) for k, v in self.flow_config.node_temperature_map.items()})
 
        llm_dict = {}
        if len(self.flow_config.clients) != 0:
            for k, v in self.flow_config.clients.items():
                if "tianchi" in v["base_url"]:
                    v['host'] = Const.HOST.value
                    v['appid'] = Const.APPID.value
                    llm_dict[k] = LLMClient(**v)

        # llm_client
        if llm_client:
            self.llm_client = llm_client
        else:
            self.llm_client = LLMClient(**self.flow_config.config["default_client"])

        # node_llm_client_map
        self.node_llm_clients: Dict[str, LLMClient] = {}
        if self.flow_config.node_llm_client_map:
            self.node_llm_clients.update({
                str(k): llm_dict[v] for k, v in self.flow_config.node_llm_client_map.items()
            })

    # ---------- generate text ----------

    # node -> client
    def _client_for(self, node_name: str) -> LLMClient:
        return self.node_llm_clients.get(node_name, self.llm_client)

    # node -> model name
    def _model_for(self, node_name: str) -> str:
        return self.node_model_map.get(node_name, Const.DEFAULT_MODEL.value)
    # node -> temperature 
    def _temperature_for(self, node_name: str) -> float:
        return self.node_temperature_map.get(node_name, 0.8)

    def _generate_text(
        self,
        node_name: str,
        prompt: str,
        max_tokens: int = 2048,
        extra_body: Dict[str, Any] = {"top_p": 0.9}
    ) -> Any:
        client = self._client_for(node_name)
        llmclient = LLMClient(
        base_url="http://10.11.175.3/tianchi/chat/completions",
        api_key='Bearer 13af0a5f5048000a72509152a644',
        timeout=60,
        host="tianchi-proxy.baidu-int.com",
        max_retry=3,
        appid='app-RNgOjXzL'
        )
        print("llmclient 创建完成")
        print("prompt", prompt)
        print("self._model_for(node_name)", self._model_for(node_name))
        print("self._temperature_for(node_name)", self._temperature_for(node_name))
        print("extra_body", extra_body)
        response =  llmclient.generate_text(
            prompt = prompt,
            model=self._model_for(node_name),
            temperature=self._temperature_for(node_name),
            extra_body = extra_body
        )
        return client.parse_chat_content(response)
    
    # --------- 统计和check工具 ----------------------
    def statistic_check_tool(self, 
                    state: ToolOptimizerState):
        """示例主函数：你可以替换为自己的 JSON 文件路径。"""
        best_record = state.get("best_record", None)
        if best_record is not None:
            self.last_tools_description_path = best_record.tool_path

        self.tools_dict = load_tools_from_json(self.last_tools_description_path) 
        statistic_check_version = "version_" + str(state.get("current_version_id", 0))
        print("当前执行的版本：", statistic_check_version)
        statistic_output = Const.PATH.value + "summary/" + statistic_check_version + "/" + self.resource_id
    
        detaildf = recall_passk_function(self.tools_dict, self.query_good_dict, self.resource_id, statistic_output)

        # 计算 top 1 的 precision
        detaildf_top1 = detaildf[(detaildf['k']==1) & (detaildf['view']=="merged")]
        relevants = detaildf_top1['gold_ids'].to_list()
        retrieveds = detaildf_top1['recall_ids'].to_list()
        precision1 = precision_at_k_batch(retrieveds, relevants, 1)
        recall1 = recall_at_k_batch(retrieveds, relevants, 1)

        # 计算 top 3 的 precision
        detaildf_top3 = detaildf[(detaildf['k']==3) & (detaildf['view']=="merged")]
        relevants = detaildf_top3['gold_ids'].to_list()
        retrieveds = detaildf_top3['recall_ids'].to_list()
        precision3 = precision_at_k_batch(retrieveds, relevants, 3)
        recall3 = recall_at_k_batch(retrieveds, relevants, 3)

        # 统计 top 3 的结果
        tools_query = {}
        tools_case_ids = []
        for indx, row in detaildf_top3.iterrows():
            recall_ids = row["recall_ids"]
            if len(recall_ids) == 0:
                recall_ids = [NO_CALL] 
            if len(tools_query.get(recall_ids[0], [])) == 0:
                tools_query[recall_ids[0]] =  []
            tools_query[recall_ids[0]].append(row["query"])
            tools_case_ids.append(recall_ids[0])
        top_tools_case = {k: tools_query[k] for k in get_k_tool(tools_case_ids, 3)}
        tools_case_all = {k: tools_query[k] for k in set(tools_case_ids)}

        version_id, next_version_id = next_version_pair(state)

        versionInfo = VersionRecord(
            version_id = statistic_check_version, 
            parent_version_id = version_id,
            stage = "statistic",
            description = self.tools_dict[self.resource_id]["description"],
            case_result = tools_case_all,
            top_case = top_tools_case,
            tool_path = statistic_output + "/tool_prompt.json",
            recall1 = recall1,
            precision1 = precision1,
            recall3 = recall3,
            precision3 = precision3
        )

        # 判断是否需要更新最佳记录
        # 条件1: best_record 不存在 (即为 None)
        # 条件2: 新的指标 (recall3, precision3) 优于旧的 best_record
        should_update = (best_record is None) or \
                (recall3 > best_record.recall3 and precision3 > best_record.precision3)

        with open(statistic_output + "/tool_prompt.json","w") as w:
            json.dump(self.tools_dict, w, ensure_ascii=False, indent=2)

        if should_update:
            return {
                    "current_version_id": version_id,
                    "next_version_id": next_version_id,
                    "version_history": append_version_history(state, versionInfo),
                    "best_description": state.get("current_description", ""),
                    "best_version_id": state.get("current_version_id", 0),
                    "best_record": versionInfo,
                    "inner_loop_cnt":0

                }
        else:
            return {
                "current_version_id": version_id,
                "next_version_id": next_version_id,
                "version_history": append_version_history(state, versionInfo),
                "inner_loop_cnt": 0
            }

    # ---------  LLMNodeCritic node--------------
    def llm_description_judge(self,  state: ToolOptimizerState):
        prompt_and_desc = LLMDescriptionJudge._gen_prompt(state, self.prompts['judge'])
        if not prompt_and_desc:
            print(f"[judge] optimizer_history 为空，跳过本轮评审")
            version_id = state.get("next_version_id", 1)
            return {
                "current_version_id": version_id,
                "next_version_id": version_id + 1,
            }
        prompt, optimizer_description = prompt_and_desc

        for _ in range(3):
            response, stype, flag = self._generate_text(node_name = "judge", prompt = prompt)
            response, flag, error_type = LLMDescriptionJudge._vertify_result(response)
            if flag:
                break
        if flag:
            relevance_score = response["relevance_score"]
            if relevance_score == 3 or  relevance_score == 2:
                self.tools_dict[self.resource_id]['description'] = optimizer_description

            judge_record = InfoRecord(
                version_id = state.get("current_version_id", "-1"),
                stage = "judge",
                info = response
            )
            return {
                "inner_loop_cnt": state.get("inner_loop_cnt", 3) + 1,
                "judge_history": append_judge_history(state, judge_record)
            }
        else:
            {
                "inner_loop_cnt": state.get("inner_loop_cnt", 3) + 1
            }
    # ---------  LLMNodeSummary node--------------
    def llm_description_summary(self, 
                                  state: ToolOptimizerState):
        prompt = LLMDescriptionSummary._gen_prompt(state, self.prompts['summary'], list(self.query_good_dict.keys()))
        print("summary prompt:", prompt)
        for _ in range(3):
            response, stype, flag = self._generate_text(node_name = "summary", prompt = prompt)
            response, flag, error_type = LLMDescriptionSummary._vertify_result(response)
            if flag:
                break
        if flag:
            optimizer_description = response["optimizer_description"]
            
            print("description: ", state.get("current_version_id", "-1"), self.tools_dict[self.resource_id]['description'])
            optimizer_record = InfoRecord(
                version_id = state.get("current_version_id", "-1"),
                stage = "summary",
                info = response
            )
            return {
                "optimizer_history": append_optimizer_history(state, optimizer_record)
            }

    # ---------- graph build ----------

def build_optimizer_graph():
    """Build and compile the LangGraph optimization workflow."""

    graph = StateGraph(ToolOptimizerState)
    graph.add_node("optimizer", optimizer)
    graph.add_node("vertify_after_optimizer", vertify_after_optimizer)
    graph.add_edge("bump_iteration", "optimizer")
    return graph.compile()


def should_continue(state: ToolOptimizerState) -> Literal["summary_optimizer", END]:
    # 达到最大轮次
    max_iteration = state.get("max_iterations", 3)
    if state.get("current_version_id", 10) > max_iteration:
        return END
    return "summary_optimizer"

# 第二层路由：judge节点结束后，控制内层循环3次
def summary_loop_router(state: ToolOptimizerState):
    if state["inner_loop_cnt"] < 3:
        return "summary_optimizer"
    # 已满3次：内层循环结束，退回统计节点做效果核验
    return "statistic_check_tool"

#------- init ----
def list_file(folder_path):
    all_items = os.listdir(folder_path)
    # 只列出文件（不包括文件夹）
    files_only = [os.path.join(folder_path, item) for item in all_items if os.path.isfile(os.path.join(folder_path, item)) and item.endswith(".pkl")]
    print("\n只列出文件:")
    return files_only


In [ ]:
!unset http_proxy
!unset https_proxy
if __name__ == "__main__":
    
    resource_ids_all = []

    for path in list_file(Const.PATH.value + "summary_output"):
        resource_ids_all.append(path.split("/")[-1].replace(".pkl", ""))

    query_good_all_dict = load_tools_from_json(Const.PATH.value +"summary/query.json")
    for resource_id, value in query_good_all_dict.items():
        if resource_id in resource_ids_all:
            print("skip !!!!", resource_id)
        print("开始执行：", resource_id)
        try:
            dag = ToolOptimizerGraph(resource_id = resource_id,
                prompt_path = "../config/prompts.json",
                flow_config_path = "../config/agent_config.yaml", # data_not_import/summary/query.json
                tools_description_path = Const.PATH.value + "summary/tool_descriptions.json",
                test_data_path  = Const.PATH.value + "summary/query.json",
                checkpointer = None)

            graph = StateGraph(ToolOptimizerState)
            graph.add_node("statistic_check_tool", dag.statistic_check_tool)
            graph.add_node("summary_optimizer", dag.llm_description_summary)
            # graph.add_node("description_judge", dag.llm_description_judge)
            graph.set_entry_point("statistic_check_tool")
            graph.add_edge("statistic_check_tool", "summary_optimizer")
            graph.add_edge("summary_optimizer", END)
            # graph.add_conditional_edges(
            #     "statistic_check_tool",
            #     should_continue,
            #     {
            #         "summary_optimizer": "summary_optimizer",
            #         END: END
            #     }
            # )
            # graph.add_edge("summary_optimizer", "description_judge")
            # # 3. 评审节点走条件路由：未满3轮继续优化；满3轮回到统计节点
            # graph.add_conditional_edges(
            #     source="description_judge",
            #     path=summary_loop_router,
            #     path_map={
            #         "summary_optimizer": "summary_optimizer",
            #         "statistic_check_tool": "statistic_check_tool"
            #     }
            # )
            compiled_graph = graph.compile()

            initial_state = {
                    "resource_id": resource_id,
                    "title": dag.tools_dict[resource_id]["title"],
                    "original_description": dag.tools_dict[resource_id]["description"],
                    # 当前工作基线指针（可手动/自动回滚）
                    "current_version_id": 0,
                    "current_description": dag.tools_dict[resource_id]["description"],
                    # 下一个版本的id
                    "next_version_id": 1,
                    # 固定参数
                    "max_iterations": 1,
                    "iteration": 0,
                    # 版本历史仓库：全量快照存储
                    "version_history": [],
                    "judge_history":[],
                    #记录历史
                    "optimizer_history": []
                }
            state = compiled_graph.invoke(initial_state)  
            print("-----------------start save state ... !!!")
            save_pickle(state, Const.PATH.value + "summary_output/" + resource_id + ".pkl")  
        except:
            print("resource_id error:", resource_id)
            continue

[原始工具描述]
【无需填写】

[Query集合]
中国人事考试网入口、tv13新闻频道直播、《深圳智慧党建》、东方财富网官方网页、湖南省电子税务局、安庆石化第一中学、北京电影学院、交管12123 app下载、中央气象台天气预报怎么查、cctv|直播、同济法学、南昌大学技术学院、中央五台直播、丹东市政府网、河北北方学院教学综合信息平台、肇兴医学院、上海动物圆、呼和浩特市招生考试信息网、上海烟草专局官网、山西省省图书馆

[输出格式]
```json
{{
"think":"按照[执行过程]逐步进行分析和思考",
"optimizer_description": "总结成一段标准、语义精简的6字左右的召回文本"
}}
```
输出:
self._model_for(node_name) deepseek-v4-flash
self._temperature_for(node_name) 0.8
extra_body {'top_p': 0.9}
llmclient 创建完成
prompt # Role
你是专业的向量召回文本优化专家。请基于[Query集合]和[原始工具描述]，提炼工具的核心语义、核心功能、适用场景、解决问题，生成一段标准化、无冗余、语义聚焦的工具召回文本。

[执行过程]
1. 理解[原始工具描述]中提供信息的核心语义和功能；
2. 分析[Query集合]中提供的应召的query的集合需要覆盖的场景和解决的问题
3. 剔除无效修饰、口语化内容、冗余话术，重点突出工具能力、使用场景、适配问题；
4. 精简工具描述适配向量相似度匹配，根据用户query优化工具描述对齐检索核心需求；
5. 执行上述过程，将工具描述总结成一段标准、语义精简的6字左右的召回文本；

[原始工具描述]
【无需填写】

[Query集合]
中国人事考试网入口、tv13新闻频道直播、《深圳智慧党建》、东方财富网官方网页、湖南省电子税务局、安庆石化第一中学、北京电影学院、交管12123 app下载、中央气象台天气预报怎么查、cctv|直播、同济法学、南昌大学技术学院、中央五台直播、丹东市政府网、河北北方学院教学综合信息平台、肇兴医学院、上海动物圆、呼和浩特市招生考试信息网、上海烟草专局官网、山西省省图书馆

[输出格式]
```json
{{
"think":"按照[执行过程]逐步

In [ ]:
!ls
# 业务参数
query = "春节祝福语"
doc = "短文"
prompt = """# Role
你是专业的向量召回文本优化专家。请基于[Query集合]和[原始工具描述]，提炼工具的核心语义、核心功能、适用场景、解决问题，生成一段标准化、无冗余、语义聚焦的工具召回文本。

[执行过程]
1. 理解[原始工具描述]中提供信息的核心语义和功能；
2. 分析[Query集合]中提供的应召的query的集合需要覆盖的场景和解决的问题
3. 剔除无效修饰、口语化内容、冗余话术，重点突出工具能力、使用场景、适配问题；
4. 精简工具描述适配向量相似度匹配，根据用户query优化工具描述对齐检索核心需求；
5. 执行上述过程，将工具描述总结成一段标准、语义精简的241字左右的召回文本；

[原始工具描述]
工具描述：该工具组件用于展示 Instagram 平台账号注册与登录入口及产品功能介绍
工具返回内容：Instagram 账号注册 / 登录指引、平台核心功能介绍
插件满足样式：
（1）官方 Instagram 标识
（2）账号注册、登录相关提示文案
（3）平台核心功能说明：拍摄、编辑、分享照片、视频以及和亲友收发消息，主打简单有趣的创意社交
插件交互样式：
（1）可进入 Instagram 账号注册或登录流程
（2）可跳转进入 Instagram 官方平台使用图片视频社交功能

[Query集合]
抖音网页端、颐和园浩雷、国家政务平台是什么、youtibe、学信网个人学历查询、metamask、kimi!、纽约时报官方网站、大众烕然、国家电网、火山云、文心一言官方app、claude、江西管理职业学院、中国保密在线、chat gpe、抖音网络版、武炼顶峰、芝商所、哔哩哔哩唧唧

[输出格式]
```json
{{
"think":"按照[执行过程]逐步进行分析和思考",
"optimizer_description": "总结成一段标准、语义精简的241字左右的召回文本"
}}
```
输出:
"""
    
    
llmclient = LLMClient(
    base_url="http://10.11.175.3/tianchi/chat/completions",
    api_key='Bearer 13af0a5f5048000a72509152a644',
    timeout=60,
    host="tianchi-proxy.baidu-int.com",
    max_retry=3,
    appid='app-RNgOjXzL'
)

response = llmclient.generate_text(prompt=prompt,
                model="deepseek-v4-flash",
                temperature=0.2,
                max_tokens=1024,
                extra_body={"top_p": 0.9},
            )
print("response", response)
print(llmclient.parse_chat_content(response))


In [3]:
!ls ../data/data_not_import/summary_output | wc -l

284
